# HGB 원인 분석 + N5·N6 개선 실험

**C1이 돌고 있는 노트북과 별개로, 새 Colab 창에서 동시에 돌릴 수 있게 만든 노트북입니다.** 결과는 `HGB_results_v3.csv`에 따로 저장돼서 기존 CSV와 섞이지 않아요.

| 단계 | 내용 | 시간 |
|---|---|---|
| R | N3 모델 재학습 (진단용 + 재현성 확인) | 약 35분 |
| D1 | 분기 픽셀 히트맵: 모델이 어떤 픽셀로 판단하나 | 1분 |
| D2 | 틀린 D·B vs 맞힌 D·B 이미지 비교 | 1분 |
| D3 | 작성자별 글자 크기·획 양 비교 | 1분 |
| N5 | 픽셀 + 모양 특징(투영·HOG) | 약 50분 |
| N6 | `max_features=0.5` (픽셀 과의존 줄이기) | 약 25분 |

**0번 셀 실행 → 세션 다운 메시지 → 1번 셀부터 끝까지 '모두 실행'이 아니라, 1번 셀을 클릭하고 '런타임 → 이후 셀 실행'**

## 0. scikit-learn 1.8.0 설치 (실행하면 세션이 재시작돼요. 정상)

In [ ]:
!pip install -q scikit-learn==1.8.0
import os; os.kill(os.getpid(), 9)

## 1. 준비 (저장소·드라이브·데이터·N3 전처리)

In [ ]:
!test -d /content/Machine_Learning || git clone --depth 1 https://github.com/ryu980920/Machine_Learning.git /content/Machine_Learning
from google.colab import drive
drive.mount('/content/drive')
import os, sys, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
OUT = '/content/drive/MyDrive/HGB_results/'; os.makedirs(OUT, exist_ok=True)
REPO = Path('/content/Machine_Learning')
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Combined/scripts'))
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Normal/scripts'))
from combined_mlp_writer_cv import load_official, load_writers
from exp006_size_com_comparison import process
import exp_cmb005_balanced_scale_stroke as L
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score

off = load_official(REPO/'02_data/datasets/MNIST_EMNIST_official_split_112000.npz')
wr  = load_writers(REPO/'02_data/Hand_made')
WRITERS = list(L.WRITERS); LABELS = '0123456789ABCDEF'
ytr, yva = off['y_train'], off['y_val']
yhm = np.concatenate([wr[w]['y'] for w in WRITERS])
whm = np.concatenate([[w]*len(wr[w]['y']) for w in WRITERS])

# N3와 같은 전처리
Xtr_aug, tw_com, _, _ = L.prepareraw(off, wr, 'scale30_stroke28')
Xva_com = process(off['X_val'], 'com', 20)
Xhm_com = np.vstack([tw_com[w] for w in WRITERS])
print(Xtr_aug.shape, Xva_com.shape, Xhm_com.shape)

## 2. 공통 함수

In [ ]:
PAIRS = [('D','0'), ('0','D'), ('B','8'), ('8','B'), ('5','F'), ('7','9')]
CSV = OUT + 'HGB_results_v3.csv'
results = []

def hm_scores(pred):
    isd = yhm < 10
    r = dict(hm_acc=accuracy_score(yhm, pred), hm_f1=f1_score(yhm, pred, average='macro'),
             hm_digit=accuracy_score(yhm[isd], pred[isd]), hm_letter=accuracy_score(yhm[~isd], pred[~isd]))
    for w in WRITERS: r[f'w{w}'] = accuracy_score(yhm[whm==w], pred[whm==w])
    for a, b in PAIRS: r[f'{a}->{b}'] = int(np.sum((yhm==LABELS.index(a)) & (pred==LABELS.index(b))))
    return r

def evaluate(name, model, Xva, Xhm, extra=None):
    pv = model.predict(Xva); ph = model.predict(Xhm)
    row = dict(name=name, val_acc=accuracy_score(yva, pv), val_f1=f1_score(yva, pv, average='macro'),
               **hm_scores(ph), **(extra or {}))
    results.append(row); pd.DataFrame(results).to_csv(CSV, index=False)
    print({k:(round(v,4) if isinstance(v,float) else v) for k,v in row.items()})
    return ph

def hgb(**kw):
    p = dict(max_iter=300, early_stopping=True, validation_fraction=0.1, n_iter_no_change=10, random_state=42)
    p.update(kw); return HistGradientBoostingClassifier(**p)

def fit(m, X, y):
    t = time.time(); m.fit(X, y); return dict(fit_s=round(time.time()-t), iters=m.n_iter_)

## R. N3 모델 재학습
같은 데이터·설정·시드라서 어젯밤 N3(손글씨 83.44%, Validation 97.58%)와 **같은 숫자가 나와야** 해요. 같으면 재현성 확인 완료.

In [ ]:
m3 = hgb(); info = fit(m3, Xtr_aug, ytr)
ph3 = evaluate('N3_재현', m3, Xva_com, Xhm_com, info)

## D1. 분기 픽셀 히트맵 — 모델이 어디를 보고 판단하나
트리의 모든 분기에서 쓰인 픽셀의 **gain(분기로 얻은 개선량) 합**을 28×28로 그려요. 밝을수록 그 픽셀에 많이 의존한다는 뜻이에요. 전체 + D·0·B·8 클래스 전용 트리를 따로 봐요.

In [ ]:
def gain_map(model, cls=None):
    g = np.zeros(784)
    for it in model._predictors:                       # 반복마다 클래스별 트리 16개
        trees = it if cls is None else [it[cls]]
        for t in trees:
            n = t.nodes[t.nodes['is_leaf'] == 0]
            np.add.at(g, n['feature_idx'], n['gain'])
    return g.reshape(28, 28)

maps = {'All': gain_map(m3)} | {c: gain_map(m3, LABELS.index(c)) for c in 'D0B8'}
fig, ax = plt.subplots(1, 5, figsize=(20, 4.5))
for a, (k, g) in zip(ax, maps.items()):
    a.imshow(g, cmap='hot'); a.set_title(f'{k} trees'); a.axis('off')
plt.suptitle('N3 HGB split-gain heatmap'); plt.tight_layout()
plt.savefig(OUT+'D1_gain_heatmap.png', dpi=130); plt.show()

# 상위 몇 개 픽셀에 의존이 몰렸는지 수치로
for k, g in maps.items():
    s = np.sort(g.ravel())[::-1]; top = s[:20].sum()/s.sum()
    used = (g > 0).sum()
    print(f'{k}: 사용 픽셀 {used}/784, 상위 20픽셀이 전체 gain의 {top:.1%}')

## D2. 틀린 D·B vs 맞힌 D·B 이미지 + 평균 이미지

In [ ]:
def show_cases(true_c, wrong_c, n=8):
    t, w = LABELS.index(true_c), LABELS.index(wrong_c)
    bad  = np.where((yhm==t) & (ph3==w))[0]; good = np.where((yhm==t) & (ph3==t))[0]
    fig, ax = plt.subplots(3, n+1, figsize=(2*(n+1), 6.5))
    rows = [(bad, f'{true_c}→{wrong_c} (wrong)'), (good, f'{true_c} correct')]
    for r, (idx, title) in enumerate(rows):
        ax[r,0].imshow(Xhm_com[idx].mean(0).reshape(28,28), cmap='gray'); ax[r,0].set_title('mean\n'+title, fontsize=8)
        for j in range(n):
            if j < len(idx):
                ax[r,j+1].imshow(Xhm_com[idx[j]].reshape(28,28), cmap='gray'); ax[r,j+1].set_title(f'w{whm[idx[j]]}', fontsize=8)
    # 3행: 틀린 평균 - 맞힌 평균 (빨강=틀린 쪽에 더 많은 획)
    diff = Xhm_com[bad].mean(0) - Xhm_com[good].mean(0)
    ax[2,0].imshow(diff.reshape(28,28), cmap='bwr', vmin=-120, vmax=120); ax[2,0].set_title('wrong − correct', fontsize=8)
    ax[2,1].imshow(Xva_com[yva==w].mean(0).reshape(28,28), cmap='gray'); ax[2,1].set_title(f'official {wrong_c} mean', fontsize=8)
    ax[2,2].imshow(Xva_com[yva==t].mean(0).reshape(28,28), cmap='gray'); ax[2,2].set_title(f'official {true_c} mean', fontsize=8)
    for a in ax.ravel(): a.axis('off')
    plt.tight_layout(); plt.savefig(OUT+f'D2_{true_c}_to_{wrong_c}.png', dpi=130); plt.show()
    print(f'{true_c}→{wrong_c}: {len(bad)}건 / 작성자별', {w_: int(np.sum(whm[bad]==w_)) for w_ in WRITERS})

show_cases('D', '0'); show_cases('B', '8')

## D3. 작성자별 글자 특성 (크기·획 양·가로세로 비율) vs 공식 자료

In [ ]:
def geom(X, th=32):
    X = X.reshape(-1, 28, 28); out = []
    for im in X:
        ys, xs = np.where(im >= th)
        if len(ys) == 0: out.append((0,0,0,0)); continue
        h, w = np.ptp(ys)+1, np.ptp(xs)+1
        out.append((max(h, w), w/h, len(ys), len(ys)/(h*w)))   # 긴 변, 가로/세로, 획 픽셀 수, 채움 비율
    return pd.DataFrame(out, columns=['long_side', 'w_over_h', 'ink_px', 'fill_ratio'])

rows = []
for name, X, y in [('official_val', off['X_val'], yva)] + [(f'writer{w}', wr[w]['X'], wr[w]['y']) for w in WRITERS]:
    g = geom(X); g['is_letter'] = y >= 10
    for part, sub in [('digit', g[~g.is_letter]), ('letter', g[g.is_letter])]:
        rows.append(dict(source=name, part=part, **sub.drop(columns='is_letter').mean().round(2).to_dict()))
geo = pd.DataFrame(rows); geo.to_csv(OUT+'D3_writer_geometry.csv', index=False); geo

## N5. 픽셀 + 모양 특징 (가로·세로 투영 56개 + HOG 324개)
**가설:** 트리는 픽셀 하나로 분기해서 획이 1픽셀만 밀려도 판단이 바뀐다. 획 방향(HOG)과 줄별 획 양(투영)은 작은 이동에 덜 흔들리므로, D의 왼쪽 직선·B의 왼쪽 세로선을 더 안정적으로 잡을 것이다.

In [ ]:
from skimage.feature import hog

def shape_feats(X):
    X = X.reshape(-1, 28, 28).astype(np.float32)
    proj = np.hstack([X.sum(1), X.sum(2)]) / 255.0                       # 세로 투영 28 + 가로 투영 28
    H = np.stack([hog(im, orientations=9, pixels_per_cell=(7,7), cells_per_block=(2,2)) for im in X])
    return np.hstack([proj, H]).astype(np.float32)

t = time.time()
F_tr, F_va, F_hm = shape_feats(Xtr_aug), shape_feats(Xva_com), shape_feats(Xhm_com)
print('특징 계산', round(time.time()-t), '초, 특징 수', F_tr.shape[1])

m5 = hgb(); info = fit(m5, np.hstack([Xtr_aug, F_tr]), ytr)
ph5 = evaluate('N5_pixel+proj+HOG', m5, np.hstack([Xva_com, F_va]), np.hstack([Xhm_com, F_hm]),
               {**info, 'n_features': 784 + F_tr.shape[1]})

## N6. `max_features=0.5` — 분기마다 픽셀 절반만 후보로
**가설:** D1에서 소수 픽셀에 gain이 몰려 있다면, 분기 후보를 무작위로 절반만 보게 해서 여러 픽셀에 판단을 분산시키면 손글씨 변형에 강해질 것이다.

In [ ]:
m6 = hgb(max_features=0.5); info = fit(m6, Xtr_aug, ytr)
ph6 = evaluate('N6_max_features=0.5', m6, Xva_com, Xhm_com, {**info, 'max_features': 0.5})

g6 = gain_map(m6); g3 = maps['All']
for k, g in [('N3', g3), ('N6', g6)]:
    s = np.sort(g.ravel())[::-1]; print(f'{k}: 상위 20픽셀 gain 비중 {s[:20].sum()/s.sum():.1%}')

## 결과 비교 + 혼동행렬

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
for a, (p, t) in zip(ax, [(ph3,'N3 (baseline)'), (ph5,'N5 pixel+shape'), (ph6,'N6 max_features=0.5')]):
    ConfusionMatrixDisplay.from_predictions(yhm, p, labels=range(16), display_labels=list(LABELS),
                                            normalize='true', values_format='.2f', ax=a, colorbar=False)
    a.set_title(t)
plt.tight_layout(); plt.savefig(OUT+'N3_N5_N6_confusion.png', dpi=120); plt.show()
pd.DataFrame(results)